# Stage 1 — English s1-style SFT on EXAONE
Use the **RTX PRO 6000 Blackwell 96GB** runtime and enable the `HF_TOKEN` secret.
Push the Stage 1 implementation to your GitHub repository before running setup.
This notebook uses a separate training environment and preserves the frozen evaluation
environment. All training settings are in `configs/stage1_sft_deepseek_pro.yaml`.
This starts from the original EXAONE base model and trains on the new Pro thinking
**plus** answer. Hyperparameters are unchanged from the original five-epoch SFT run.
The published HF dataset must contain the two Pro columns before you run preparation.
Its current revision is resolved to an immutable commit and recorded in the run manifest.
Checkpoints use the separate run name `sft_s1k_deepseek_pro`.
This notebook performs SFT and saves all five epoch checkpoints. It never runs evaluation.
A default Run all prepares and inspects data only; enable `RUN_TRAINING` to train.
Keep enough Drive space for five full model/optimizer checkpoints (allow at least 100GB).
For later evaluation, set the evaluation notebook's CONFIG to
`configs/stage1_sft_deepseek_pro.yaml` to select these new checkpoints.

In [ ]:
REPO_URL = "https://github.com/seungjun-green/post-training-aime.git"
CODE_ROOT = "/content/lg-aime-stage1"
TRAIN_ROOT = "/content/drive/MyDrive/LG-AIME-Stage1"
TRAIN_ENV = "/content/lg-sft-env"
CONFIG = "configs/stage1_sft_deepseek_pro.yaml"

In [ ]:
import os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
if not Path(CODE_ROOT).exists():
    subprocess.check_call(["git", "clone", "--branch", "main", REPO_URL, CODE_ROOT])
def git(*args):
    return subprocess.check_output(["git", "-C", CODE_ROOT, *args], text=True).strip()
if git("remote", "get-url", "origin") != REPO_URL or git("status", "--porcelain"):
    raise ValueError("Use a clean checkout of the configured repository")
if git("branch", "--show-current") != "main":
    raise ValueError("Use a main checkout")
subprocess.check_call(["git", "-C", CODE_ROOT, "pull", "--ff-only", "origin", "main"])
print("Code commit:", git("rev-parse", "HEAD"))
if not (Path(CODE_ROOT) / CONFIG).is_file():
    raise FileNotFoundError(
        f"The GitHub checkout does not contain {CONFIG}. Commit and push the Pro SFT "
        "update (including configs/stage1_sft_deepseek_pro.yaml, train/sft_data.py, "
        "and train/stage1_sft.py) to main, then rerun this setup cell. "
        "Uploading the notebook alone does not update the cloned training code."
    )
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "uv==0.11.22"])
subprocess.check_call([sys.executable, "scripts/setup_stage1_runtime.py", "--venv", TRAIN_ENV], cwd=CODE_ROOT)
sys.path.insert(0, CODE_ROOT)
from common.process import run_logged
TRAIN_COMMAND = [str(Path(TRAIN_ENV) / "bin/python"), "train/stage1_sft.py",
                 "--config", CONFIG, "--output_root", TRAIN_ROOT]

## Data and loss-mask inspection
This reads all 996 published rows without loading model weights. It skips missing Pro
outputs and tokenizes the new reasoning plus answer with the existing English prompt
and native chat template. Review the printed full text and masked text. Full sequences
longer than 20,480 tokens are dropped, never truncated. For this uploaded export, expect
718 retained rows, six missing-output exclusions, and 272 overlength exclusions.
Old grades describe the old answers and never filter training. Training repeats these
checks before the first optimizer step.

In [ ]:
run_logged(TRAIN_COMMAND + ["--prepare-only"], cwd=CODE_ROOT,
           log_path=Path(TRAIN_ROOT) / "prepare_console.log")

## Five-epoch full fine-tuning
Each complete epoch is saved with its tokenizer, model code, optimizer, scheduler and RNG.
Set `RESUME_CHECKPOINT` to the latest complete epoch directory to resume an interrupted run.
Keep the same code commit and config. An incomplete epoch is replayed from its start.
An existing run will not be silently overwritten. Do not rerun setup during training.

In [ ]:
RUN_TRAINING = False
RESUME_CHECKPOINT = None
if RUN_TRAINING:
    command = TRAIN_COMMAND + (["--resume_from_checkpoint", RESUME_CHECKPOINT] if RESUME_CHECKPOINT else [])
    run_logged(command, cwd=CODE_ROOT, log_path=Path(TRAIN_ROOT) / "training_console.log")
else:
    print("Training is off. Review preparation, then enable RUN_TRAINING.")